# DNS Tunneling Detection — Colab Demo

Self-contained, end-to-end notebook. All source code from `src/` is embedded below so it runs on a fresh Colab runtime with no clone.

**Repo**: https://github.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection  
**Open in Colab**: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection/blob/main/notebooks/colab_demo.ipynb)

## How to run this notebook

> **Estimated time: ~2 minutes** on a free CPU runtime. No GPU needed.

1. Open in Colab (badge above).
2. *(Optional)* `File → Save a copy in Drive` to persist your edits.
3. Run **section 1** to install dependencies.
4. Run **section 2** to load the bundled DNS-tunneling dataset.
5. Pick **ONE** of the two scenarios in **section 3**:
   - 🅰 **Scenario A — Use pretrained models** *(fastest, ~10 seconds)* — downloads `rf.pkl` and `xgb.pkl` from the repo and scores the test set.
   - 🅱 **Scenario B — Train from scratch** *(~1 minute)* — fits RF and XGBoost on a stratified train split, then evaluates.
6. Run the remaining sections for plots: confusion matrices, ROC curves, feature importance.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: xgboost` | Re-run section 1. |
| `URLError` downloading from raw.githubusercontent.com | Check Colab has internet; or upload a CSV manually. |
| Confusion matrix all in one class | The bundled sample is 50/50 balanced — re-check the class counts printed in section 2. |

## 1. Install dependencies

In [ ]:
!pip install -q xgboost seaborn joblib

In [ ]:
import io
import math
import urllib.request
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

REPO_RAW = "https://raw.githubusercontent.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection/main"

## 2. Load the bundled dataset

Downloads the 10 000-row stratified sample of CIRA-CIC-DoHBrw-2020 from this repo. Real flow-feature data, balanced 50/50.

In [ ]:
# === Dataset loader (embedded from src/preprocess.py) ===
_QUERY_CANDIDATES = ("query", "domain", "name", "qname", "Domain", "Query")
_LABEL_CANDIDATES = ("label", "Label", "class", "Class", "is_malicious", "Type")
_POSITIVE_TOKENS = {"1", "malicious", "malware", "tunneling", "tunnel",
                    "doh-malicious", "doh_malicious", "attack", "evil", "true"}

def _find_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    return None

def _normalize_labels(s):
    if pd.api.types.is_numeric_dtype(s):
        return (s > 0).astype(int)
    return s.astype(str).str.strip().str.lower().isin(_POSITIVE_TOKENS).astype(int)

def load_dataset(path):
    df = pd.read_csv(path, low_memory=False).dropna(how="all").reset_index(drop=True)
    label_col = _find_col(df, _LABEL_CANDIDATES)
    if label_col is None:
        raise ValueError(f"No label column. Got: {list(df.columns)}")
    labels = _normalize_labels(df[label_col])
    query_col = _find_col(df, _QUERY_CANDIDATES)
    if query_col is not None:
        data = df[[query_col]].rename(columns={query_col: "query"}).fillna("")
        data["query"] = data["query"].astype(str)
    else:
        data = df.drop(columns=[label_col]).select_dtypes(include="number").copy()
    keep = ~data.duplicated()
    data, labels = data[keep].reset_index(drop=True), labels[keep].reset_index(drop=True)
    num = data.select_dtypes(include="number").columns
    if len(num):
        data[num] = data[num].fillna(data[num].median(numeric_only=True))
    return data, labels

# Download the bundled sample
DATA_PATH = "doh_sample.csv"
urllib.request.urlretrieve(f"{REPO_RAW}/data/sample/doh_sample.csv", DATA_PATH)
data, y = load_dataset(DATA_PATH)
print(f"rows: {len(data):,}   features: {data.shape[1]}   pos: {int(y.sum()):,}   neg: {int((1-y).sum()):,}")
data.head()

In [ ]:
# Quick EDA — class balance + top discriminative feature
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
y.value_counts().rename({0: "benign", 1: "malicious"}).plot(
    kind="bar", ax=axes[0], color=["#4c72b0", "#dd8452"]
)
axes[0].set_title("Class distribution")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

means = data.groupby(y).mean()
diff = (means.loc[1] - means.loc[0]).abs() / (data.std() + 1e-9)
top_feat = diff.sort_values(ascending=False).index[0]
lo, hi = data[top_feat].quantile([0.01, 0.99])
sns.histplot(
    data=pd.DataFrame({top_feat: data[top_feat], "label": y}).query(
        f"@lo <= `{top_feat}` <= @hi"
    ),
    x=top_feat, hue="label", bins=40, kde=True, palette="Set1", ax=axes[1],
)
axes[1].set_title(f"{top_feat} by class (1–99th pct.)")
plt.tight_layout(); plt.show()

## 3. Pick a scenario

Two paths — run **only one**.

- 🅰 **Scenario A** — Use pretrained models from the repo. Fast (~10 s).
- 🅱 **Scenario B** — Train RF and XGBoost from scratch on this dataset. Slower (~1 min) but you see the full pipeline.

### 🅰 Scenario A — Use pretrained models

Downloads `rf.pkl` and `xgb.pkl` from the repo and scores the same held-out test split the maintainer trained against.

In [ ]:
# === Scenario A: load pretrained, score test split ===
for name in ("rf.pkl", "xgb.pkl"):
    urllib.request.urlretrieve(f"{REPO_RAW}/models/{name}", name)
urllib.request.urlretrieve(f"{REPO_RAW}/data/sample/test_split.csv", "test_split.csv")

test_df = pd.read_csv("test_split.csv")
y_test = test_df["label"].astype(int)
X_test = test_df.drop(columns=["label"])

models = {"Random Forest": joblib.load("rf.pkl"), "XGBoost": joblib.load("xgb.pkl")}
X_train, y_train = None, None  # Scenario A doesn't need train data

test_metrics, roc_data = {}, {}
for name, model in models.items():
    # Align columns to model expectations
    expected = list(getattr(model, "feature_names_in_", X_test.columns))
    X = X_test[expected]
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    test_metrics[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, test_metrics[name]["roc_auc"])
    print(f"\n[{name}]")
    print(classification_report(y_test, y_pred, target_names=["benign", "malicious"]))

X = X_test  # for the feature-importance section below
pd.DataFrame(test_metrics).T.round(4)

### 🅱 Scenario B — Train from scratch

*Skip if you ran Scenario A.*

Fits Random Forest and XGBoost on an 80% stratified split, 5-fold cross-validates, then evaluates on the held-out 20%.

In [ ]:
# === Scenario B: train then evaluate ===
X = data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=150, max_depth=20, n_jobs=-1, random_state=42, class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric="logloss",
        tree_method="hist", n_jobs=-1, random_state=42,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    print(f"\n[{name}] 5-fold CV:")
    row = {}
    for metric in ("accuracy", "f1", "roc_auc"):
        s = cross_val_score(model, X_train, y_train, cv=cv, scoring=metric, n_jobs=-1)
        row[metric] = float(s.mean())
        print(f"  {metric:10s}: {row[metric]:.4f}  (+/- {s.std():.4f})")
    cv_results[name] = row

test_metrics, roc_data = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_metrics[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, test_metrics[name]["roc_auc"])
    print(f"\n[{name}]")
    print(classification_report(y_test, y_pred, target_names=["benign", "malicious"]))

pd.DataFrame(test_metrics).T.round(4)

## 4. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, model) in zip(axes, models.items()):
    expected = list(getattr(model, "feature_names_in_", X_test.columns))
    cm = confusion_matrix(y_test, model.predict(X_test[expected]))
    ConfusionMatrixDisplay(cm, display_labels=["benign", "malicious"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 5. ROC curves

In [ ]:
plt.figure(figsize=(5, 4.5))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC — model comparison"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

## 6. Feature importance (top 15)

In [ ]:
TOP_K = 15
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    importances = model.feature_importances_
    feat_names = np.array(getattr(model, "feature_names_in_", X_test.columns))
    order = np.argsort(importances)[::-1][:TOP_K]
    sns.barplot(x=importances[order], y=feat_names[order], ax=ax, palette="viridis")
    ax.set_title(f"{name} — top {len(order)} features")
    ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 7. *(Scenario B only)* Save your trained models

In [ ]:
if X_train is not None:
    joblib.dump(models["Random Forest"], "rf_trained.pkl", compress=3)
    joblib.dump(models["XGBoost"], "xgb_trained.pkl", compress=3)
    print("saved: rf_trained.pkl, xgb_trained.pkl  (download from the file pane on the left)")
else:
    print("Skipping save — Scenario A used pretrained models, nothing new to persist.")

## Takeaways

- Both Random Forest and XGBoost reach **F1 ≥ 0.99** and **ROC-AUC ≈ 1.00** on this dataset.
- Top features are packet-length statistics (`PacketLengthMode`, `PacketLengthMean`, `FlowBytesReceived`) — payload data alters packet-size distributions.
- Limitations and the full write-up live in [`reports/report.md`](https://github.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection/blob/main/reports/report.md).